# The UnderCover Agent on AgentCore Runtime

The UnderCover Agent is a LangGraph agent on OpenRouter. Its tools are the UnderCover app's operations, reached through the AgentCore Gateway, so the rule on payouts applies to it. This notebook deploys the agent to AgentCore Runtime with the AgentCore CLI, talks to it with boto3, reads its logs, then tears it down.

The agent's code is in `undercover/app/undercover_agent/`, and `undercover/agentcore/agentcore.json` describes the deployment. The CLI turns that description into one CloudFormation stack.

What the deploy creates:

- a CloudFormation stack, `AgentCore-undercover-default`, with the runtime and the role it runs as
- a credential provider, `undercoverOpenAI`, which keeps the OpenRouter key in AgentCore Identity
- an AWS KMS key. The CLI moves your account's AgentCore token vault onto a key of its own whenever a project holds an API key
- a log group for the runtime, which AgentCore makes on the first request
- the first time only, the CDK's own `CDKToolkit` stack: a bucket, a container repository, five roles and a second KMS key

Runtime bills for the CPU and memory an agent uses while it works, which is cents here. The two KMS keys are the standing cost, $1 a month each, charged by the hour, so the teardown at the end matters.

Run `lambda_deploy.ipynb` and `gateway.ipynb` first.

## 1. Setup

The AgentCore CLI is a Node program, installed once with `npm install -g @aws/agentcore`.

In [ ]:
import json
import os
import time
import uuid
from pathlib import Path

import boto3
from dotenv import load_dotenv

PREFIX = "undercover"
GATEWAY_NAME = "undercover-gateway"
PROJECT_DIR = Path("undercover")
STACK_NAME = "AgentCore-undercover-default"
CLI_KEY_PROVIDER = "undercoverOpenAI"
CLI_KEY_DESCRIPTION = "AgentCore Identity encryption key for undercover"
PROJECT_TAG = {"project": "undercover"}

session = boto3.Session()
region = session.region_name
agentcore = session.client("bedrock-agentcore-control")
agentcore_data = session.client("bedrock-agentcore")
cloudformation = session.client("cloudformation")
iam = session.client("iam")
kms = session.client("kms")
logs = session.client("logs")
tagging = session.client("resourcegroupstaggingapi")

identity = session.client("sts").get_caller_identity()
account = identity["Account"]
print(f"Working as {identity['Arn']} in {region}")

!agentcore --version

The searches this notebook uses to find what it made, by name and by description. They are here at the top so that the teardown at the end works on another day with only this section run.

In [ ]:
def find_runtimes():
    pages = agentcore.get_paginator("list_agent_runtimes").paginate()
    return [r for page in pages for r in page["agentRuntimes"] if r["agentRuntimeName"].startswith(PREFIX)]


def find_runtime_log_groups():
    pages = logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix=f"/aws/bedrock-agentcore/runtimes/{PREFIX}")
    return [g["logGroupName"] for page in pages for g in page["logGroups"]]


def find_stacks():
    pages = cloudformation.get_paginator("describe_stacks").paginate()
    return [s["StackName"] for page in pages for s in page["Stacks"] if s["StackName"] in (STACK_NAME, "CDKToolkit")]


def find_key_providers():
    pages = agentcore.get_paginator("list_api_key_credential_providers").paginate()
    return [p["name"] for page in pages for p in page["credentialProviders"] if p["name"].startswith(PREFIX)]


def find_roles():
    pages = iam.get_paginator("list_roles").paginate()
    return [r["RoleName"] for page in pages for r in page["Roles"] if r["RoleName"].startswith((PREFIX, "AgentCore-", "cdk-"))]


def find_customer_keys():
    """Every KMS key of yours that is in use, with what it is for."""
    aliases = {a.get("TargetKeyId"): a["AliasName"] for page in kms.get_paginator("list_aliases").paginate() for a in page["Aliases"]}
    found = {}
    for page in kms.get_paginator("list_keys").paginate():
        for key in page["Keys"]:
            metadata = kms.describe_key(KeyId=key["KeyId"])["KeyMetadata"]
            if metadata["KeyManager"] == "CUSTOMER" and metadata["KeyState"] == "Enabled":
                found[metadata["KeyId"]] = metadata.get("Description") or aliases.get(metadata["KeyId"], "")
    return found


def find_tagged():
    tag_filter = [{"Key": key, "Values": [value]} for key, value in PROJECT_TAG.items()]
    pages = tagging.get_paginator("get_resources").paginate(TagFilters=tag_filter)
    return [r["ResourceARN"] for page in pages for r in page["ResourceTagMappingList"]]


def show_resources():
    vault = agentcore.get_token_vault(tokenVaultId="default")["kmsConfiguration"]
    found = {
        "Stacks": find_stacks(),
        "Runtimes": [r["agentRuntimeName"] for r in find_runtimes()],
        "API key providers": find_key_providers(),
        "Runtime log groups": find_runtime_log_groups(),
        "IAM roles": find_roles(),
        "KMS keys in use": [f"{key_id[:8]} ({purpose})" for key_id, purpose in find_customer_keys().items()],
        "Token vault key": [vault.get("kmsKeyArn", "the AWS key").split("/")[-1][:8] if vault["keyType"] == "CustomerManagedKey" else "the AWS key"],
        "Tagged project=undercover": find_tagged(),
    }
    for label, names in found.items():
        print(f"{label + ':':<28}{', '.join(names) or 'none'}")

A look at the budget before anything is deployed.

In [ ]:
budgets = session.client("budgets").describe_budgets(AccountId=account).get("Budgets", [])

for budget in budgets:
    limit = float(budget["BudgetLimit"]["Amount"])
    actual = float(budget["CalculatedSpend"]["ActualSpend"]["Amount"])
    print(f"{budget['BudgetName']}: ${actual:,.2f} spent of ${limit:,.2f}")

if not budgets:
    print("No budget found. Create one in Billing and Cost Management, Budgets, before deploying.")

## 2. Point the project at the gateway

Two things in the project depend on your account, and this cell writes them.

- `agentcore/.env.local` holds the OpenRouter key for the CLI, which stores it in AgentCore Identity at deploy time. The file is ignored by git.
- `agentcore.json` gives the runtime one environment variable, the gateway's address.

The gateway is found by name. Its address changes whenever it is recreated, so run this cell again, and deploy again, after a new run of `gateway.ipynb`.

One more file is part of the wiring and needs no change: `gateway_policy.json`, beside the agent's code, lets the runtime's role call any gateway of yours named `undercover-gateway`. `agentcore.json` lists it under `additionalPolicies`.

In [ ]:
load_dotenv()

gateway_summary = next(
    g for page in agentcore.get_paginator("list_gateways").paginate() for g in page["items"] if g["name"] == GATEWAY_NAME
)
gateway = agentcore.get_gateway(gatewayIdentifier=gateway_summary["gatewayId"])

env_local = PROJECT_DIR / "agentcore" / ".env.local"
env_local.write_text(
    f"AGENTCORE_CREDENTIAL_UNDERCOVEROPENAI={os.environ['OPENROUTER_API_KEY']}\n"
    f"UNDERCOVER_GATEWAY_URL={gateway['gatewayUrl']}\n"
)

config_path = PROJECT_DIR / "agentcore" / "agentcore.json"
config = json.loads(config_path.read_text())
config["runtimes"][0]["envVars"] = [{"name": "UNDERCOVER_GATEWAY_URL", "value": gateway["gatewayUrl"]}]
config_path.write_text(json.dumps(config, indent=2) + "\n")

print(gateway["gatewayUrl"])
print(json.dumps(config["runtimes"][0], indent=2))

## 3. Try it on your machine first

The CLI runs the same agent locally. In a terminal, from the `undercover` folder:

```bash
agentcore dev --logs --skip-deploy
```

and from a second terminal:

```bash
agentcore dev "Look up claim CLM-1002"
```

`--skip-deploy` matters, because `agentcore dev` deploys to AWS first without it. The local agent calls the real gateway and signs its requests with your own AWS credentials. On Runtime the same code signs with the runtime's role.

## 4. Deploy

`agentcore deploy` stores the OpenRouter key, builds a zip of the agent's code with its dependencies for Linux, and deploys the stack through the CDK. The first deploy in an account also bootstraps the CDK, which adds a minute and a half. A redeploy after a code change takes about a minute.

In [ ]:
!cd {PROJECT_DIR} && agentcore deploy --yes

## 5. Talk to the agent

`invoke_agent_runtime` is the whole client side. A session id keeps a conversation on one microVM, which AgentCore starts on the first request of a session and stops after 60 idle seconds, the setting in `agentcore.json`. The first request of a session waits for that start.

In [ ]:
runtime = find_runtimes()[0]
print(runtime["agentRuntimeName"], runtime["status"])


def ask(prompt, session_id):
    start = time.perf_counter()
    response = agentcore_data.invoke_agent_runtime(
        agentRuntimeArn=runtime["agentRuntimeArn"],
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    answer = json.loads(response["response"].read())["result"]
    print(f"[{time.perf_counter() - start:.1f}s] {prompt}")
    print(answer)
    print()


session_id = f"undercover-session-{uuid.uuid4().hex}"
ask("Look up claim CLM-1002 and tell me who the customer is and the amount claimed.", session_id)

The rule at the gateway applies to the agent as it did to the notebook. The agent's code turns the gateway's refusal into a tool result, so the agent can say what happened. The follow-up question shows that the session remembers the conversation.

In [ ]:
ask("Pay claim CLM-1002 in full: 1150 dollars.", session_id)
ask("What happened with that payout, and what was the claim for?", session_id)
ask("Pay 420 dollars on claim CLM-1001.", session_id)

## 6. The agent's logs

AgentCore writes the agent's logs to a log group named after the runtime. The agent logs a line whenever the gateway refuses a call.

In [ ]:
time.sleep(10)

for log_group in find_runtime_log_groups():
    pages = logs.get_paginator("filter_log_events").paginate(
        logGroupName=log_group, filterPattern='"Gateway refused"', startTime=int((time.time() - 900) * 1000)
    )
    messages = {json.loads(e["message"])["message"] for page in pages for e in page["events"] if e["message"].startswith('{"timestamp"')}
    print(log_group)
    for message in messages:
        print(f"    {message}")

## 7. What now exists in the account

Found by name and by description, then by the `project=undercover` tag.

In [ ]:
show_resources()

## 8. Teardown (optional)

Run this when you are done. The order matters.

1. The stack goes first, which removes the runtime and its role.
2. The credential provider and the runtime's log group are outside the stack, so they are deleted by name.
3. The token vault goes back to the AWS key. Only then is the CLI's KMS key scheduled for deletion. A key that the vault still uses must never be deleted, because every credential in the vault would become unreadable.

KMS deletes a key after a waiting period of at least seven days. It stops billing as soon as the deletion is scheduled, and `cancel_key_deletion` brings it back within that time.

In [ ]:
if STACK_NAME in find_stacks():
    cloudformation.delete_stack(StackName=STACK_NAME)
    cloudformation.get_waiter("stack_delete_complete").wait(StackName=STACK_NAME, WaiterConfig={"Delay": 10})
    print(f"Deleted stack {STACK_NAME}")

if CLI_KEY_PROVIDER in find_key_providers():
    agentcore.delete_api_key_credential_provider(name=CLI_KEY_PROVIDER)
    print(f"Deleted credential provider {CLI_KEY_PROVIDER}")

for log_group in find_runtime_log_groups():
    logs.delete_log_group(logGroupName=log_group)
    print(f"Deleted log group {log_group}")

In [ ]:
cli_keys = [key_id for key_id, purpose in find_customer_keys().items() if purpose == CLI_KEY_DESCRIPTION]
vault = agentcore.get_token_vault(tokenVaultId="default")["kmsConfiguration"]

if vault["keyType"] == "CustomerManagedKey" and vault["kmsKeyArn"].split("/")[-1] in cli_keys:
    agentcore.set_token_vault_cmk(tokenVaultId="default", kmsConfiguration={"keyType": "ServiceManagedKey"})
    print("The token vault is back on the AWS key")

vault = agentcore.get_token_vault(tokenVaultId="default")["kmsConfiguration"]

for key_id in cli_keys:
    if vault.get("kmsKeyArn", "").endswith(key_id):
        print(f"Key {key_id[:8]} is still the vault's key, left alone")
        continue
    deletion = kms.schedule_key_deletion(KeyId=key_id, PendingWindowInDays=7)
    print(f"Key {key_id[:8]} will be deleted on {deletion['DeletionDate']:%Y-%m-%d}")

show_resources()

## 9. Remove the CDK bootstrap (optional)

The `CDKToolkit` stack is shared by every CDK project in this account and region. Leave it if anything else of yours uses the CDK here. It is worth removing otherwise, because its KMS key costs $1 a month. The next `agentcore deploy` makes it again.

The assets bucket keeps old versions of every file and outlives its stack, so the cell empties it, deletes the stack, then deletes the bucket.

In [ ]:
REMOVE_CDK_BOOTSTRAP = False

if REMOVE_CDK_BOOTSTRAP and "CDKToolkit" in find_stacks():
    bucket = session.resource("s3").Bucket(f"cdk-hnb659fds-assets-{account}-{region}")
    bucket.object_versions.delete()
    cloudformation.delete_stack(StackName="CDKToolkit")
    cloudformation.get_waiter("stack_delete_complete").wait(StackName="CDKToolkit", WaiterConfig={"Delay": 10})
    bucket.delete()
    print("Deleted the CDKToolkit stack and its bucket")

show_resources()